# LLM Classification Finetuning – Baseline using TF-IDF + RandomForest

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

# Load data
train = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")
test = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")

train.head()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Combine prompt and responses into one string
train['text'] = train['prompt'] + ' ' + train['response_a'] + ' ' + train['response_b']

# Vectorize with TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(train['text'])

# Create labels from one-hot target columns
y = np.argmax(train[['winner_model_a', 'winner_model_b', 'winner_tie']].values, axis=1)

# Train a simple model
clf = RandomForestClassifier()
clf.fit(X, y)



In [ ]:
import pandas as pd

# Load the test.csv file
test = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')

# View the first few rows to confirm
test.head()


In [ ]:
# Combine prompt + response_a + response_b into a single text input
test['text'] = test['prompt'] + ' ' + test['response_a'] + ' ' + test['response_b']

# View it
test[['id', 'text']].head()

In [ ]:
# Transform test text
X_test = vectorizer.transform(test['text'])

# Check shape (just to verify)
X_test.shape

In [ ]:
# Predict class probabilities
probs = clf.predict_proba(X_test)

# Check a few example rows
probs[:5]

In [ ]:
# Create the submission DataFrame
submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': probs[:, 0],
    'winner_model_b': probs[:, 1],
    'winner_tie': probs[:, 2],
})

# Save to CSV
submission.to_csv('submission.csv', index=False)

# Preview
submission.head()
